# **Coleta de Súmulas CBF — 2020–2026**

## **1. Bibliotecas**

Além das dependências padrão, este notebook usa:
- `requests` + `urllib3` — requisições HTTP com retry automático e desabilitação de warnings SSL
- `pandas` — construção e exportação dos índices em CSV

In [1]:
import json
import re
import time
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import requests
import urllib3
from urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(InsecureRequestWarning)

## **2. Configuração**

Define os caminhos de saída e os metadados das três competições coletadas: **Série A**, **Série B** e **Copa do Brasil** (2020–2026).

Os IDs de competição da Série A são fixos e conhecidos. Os da Série B e Copa do Brasil são descobertos automaticamente na seção 4 e salvos em cache para evitar requisições repetidas em execuções futuras.

In [ ]:
BASE_URL = "https://www.cbf.com.br"
API_URL = f"{BASE_URL}/api/cbf/jogos/campeonato"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
MATCH_REPORTS_PATH = RAW_DATA_PATH / "match-reports"
VIZ_PATH = PROJECT_ROOT / "viz"
COMPETITION_IDS_CACHE = DATA_PATH / "competition_ids_cache.json"

YEARS = list(range(2020, 2027))
REQUEST_DELAY = 0.2

COMPETITIONS = {
    "serie-a": {
        "name": "Campeonato Brasileiro Serie A",
        "type": "league",
        "max_rounds": 38,
        "ids": {
            2020: "12464",
            2021: "12487",
            2022: "12518",
            2023: "12555",
            2024: "12584",
            2025: "12606",
            2026: "1260611",
        },
    },
    "serie-b": {
        "name": "Campeonato Brasileiro Serie B",
        "type": "league",
        "max_rounds": 38,
        "ids": {year: None for year in range(2020, 2027)},
    },
    "copa-do-brasil": {
        "name": "Copa do Brasil",
        "type": "cup",
        "max_rounds_per_phase": 2,
        "tabela_url": f"{BASE_URL}/futebol-brasileiro/tabelas/copa-do-brasil/masculino",
        "ids": {year: None for year in range(2020, 2027)},
    },
}

for path in [MATCH_REPORTS_PATH, VIZ_PATH]:
    path.mkdir(parents=True, exist_ok=True)

## **3. Funções Utilitárias**

- `normalize_text` — remove acentos e converte para minúsculas; base para comparações robustas de strings.
- `slugify` — transforma nome de time em slug para compor nomes de arquivo.
- `safe_int` — converte campos numéricos com fallback seguro para dados faltantes da API.
- `fetch_json` — GET com retry exponencial; lança `RuntimeError` após esgotar tentativas.

In [ ]:
def normalize_text(value: str) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(char for char in text if not unicodedata.combining(char))
    return text.casefold().strip()


def slugify(value: str) -> str:
    text = normalize_text(value)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-") or "unnamed"


def safe_int(value, default: int = 0) -> int:
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return default


def fetch_json(session: requests.Session, url: str, max_retries: int = 3) -> Optional[dict]:
    last_error = None
    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            last_error = error
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
    raise RuntimeError(f"Failed: {url}") from last_error

## **4. Descoberta de IDs das Competições**

A API da CBF identifica cada competição por um `competition_id` numérico. Os IDs da Série A são fixos e hardcoded. Para as demais competições, dois mecanismos de auto-descoberta são usados:

- **Série B** — busca em torno do ID da Série A do mesmo ano; valida que o candidato tem ≥ 8 jogos nas rodadas 1 e 38.
- **Copa do Brasil** — raspa a página de tabela da CBF para extrair os IDs de cada fase via regex.

Os IDs encontrados são salvos em `data/competition_ids_cache.json`. Execuções subsequentes carregam do cache instantaneamente.

In [ ]:
def _count_games(session: requests.Session, comp_id: str, round_number: int) -> int:
    url = f"{API_URL}/{comp_id}/rodada/{round_number}/fase"
    try:
        r = session.get(url, timeout=10)
        if r.status_code != 200:
            return 0
        payload = r.json()
        groups = payload if isinstance(payload, list) else (payload or {}).get("jogos", []) or []
        return sum(len(g.get("jogo") or []) for g in groups)
    except Exception:
        return 0


def _discover_serie_b_ids(session: requests.Session, search_radius: int = 35) -> None:
    for year in YEARS:
        if COMPETITIONS["serie-b"]["ids"].get(year) is not None:
            continue
        base_id_str = COMPETITIONS["serie-a"]["ids"].get(year)
        if not base_id_str:
            continue
        base_id = int(base_id_str)
        print(f"  {year}: scanning ±{search_radius} around Serie A={base_id_str}...")
        for offset in range(-search_radius, search_radius + 1):
            comp_id = str(base_id + offset)
            if comp_id == base_id_str:
                continue
            n_r1 = _count_games(session, comp_id, 1)
            if n_r1 < 8:
                time.sleep(REQUEST_DELAY)
                continue
            time.sleep(REQUEST_DELAY)
            n_r38 = _count_games(session, comp_id, 38)
            time.sleep(REQUEST_DELAY)
            if n_r38 >= 8:
                COMPETITIONS["serie-b"]["ids"][year] = comp_id
                print(f"    Serie B → {comp_id}")
                break
        else:
            print(f"    Serie B → not found for {year}")


def _discover_copa_phase_ids(session: requests.Session) -> None:
    tabela_url = COMPETITIONS["copa-do-brasil"]["tabela_url"]
    for year in YEARS:
        if COMPETITIONS["copa-do-brasil"]["ids"].get(year) is not None:
            continue
        url = f"{tabela_url}/{year}"
        try:
            r = session.get(url, timeout=30)
            if r.status_code != 200:
                print(f"    Copa do Brasil {year} → HTTP {r.status_code}")
                continue
            pattern = rf'/futebol-brasileiro/tabelas/copa-do-brasil/masculino/{year}/(\d+)'
            phase_ids = list(dict.fromkeys(re.findall(pattern, r.text)))
            if phase_ids:
                COMPETITIONS["copa-do-brasil"]["ids"][year] = phase_ids
                print(f"    Copa do Brasil {year} → {len(phase_ids)} phases: {phase_ids}")
            else:
                print(f"    Copa do Brasil {year} → no phases found in HTML")
        except Exception as e:
            print(f"    Copa do Brasil {year} → error: {e}")
        time.sleep(REQUEST_DELAY)


def _load_ids_from_cache() -> bool:
    if not COMPETITION_IDS_CACHE.exists():
        return False
    with open(COMPETITION_IDS_CACHE) as f:
        cache = json.load(f)
    for slug, ids in cache.items():
        if slug not in COMPETITIONS:
            continue
        for year_str, comp_id in ids.items():
            year = int(year_str)
            if year in COMPETITIONS[slug]["ids"]:
                COMPETITIONS[slug]["ids"][year] = comp_id
    return True


def _save_ids_to_cache() -> None:
    cache = {
        slug: {str(y): id_ for y, id_ in comp["ids"].items() if id_ is not None}
        for slug, comp in COMPETITIONS.items()
    }
    COMPETITION_IDS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    with open(COMPETITION_IDS_CACHE, "w") as f:
        json.dump(cache, f, indent=2)


def discover_all_ids() -> None:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False

    print("Discovering Serie B IDs...")
    _discover_serie_b_ids(session)

    print("Discovering Copa do Brasil phase IDs...")
    _discover_copa_phase_ids(session)

    _save_ids_to_cache()
    print(f"Cache saved to: {COMPETITION_IDS_CACHE}")


print("Loading competition IDs...")
if _load_ids_from_cache():
    print("  Loaded from cache.")
else:
    print("  Cache not found — discovering...")
    discover_all_ids()
print("Done.")

## **5. Coleta de Documentos**

Para cada competição e ano, `collect_documents` percorre todos os rounds via API e coleta os metadados de todos os documentos disponíveis por jogo (súmula, relatório, boletim financeiro). A Copa do Brasil usa scraping da página de tabela, pois seus jogos não estão na API de rounds.

O resultado é salvo em `cbf_documents_2020_2026.csv`. A tabela de resumo abaixo confirma cobertura de todos os anos e competições.

In [ ]:
def flatten_documents(
    payload: dict,
    year: int,
    competition_id: str,
    round_number: int,
    competition: str = "",
    competition_name: str = "",
) -> List[Dict[str, str]]:
    rows = []
    if isinstance(payload, list):
        groups = payload
    else:
        groups = (payload or {}).get("jogos", []) or []
    for group in groups:
        for game in (group.get("jogo", []) or []):
            base = {
                "competition": competition,
                "competition_name": competition_name,
                "year": year,
                "competition_id": competition_id,
                "round": safe_int(game.get("rodada"), round_number),
                "game_id": game.get("id_jogo"),
                "game_number": safe_int(game.get("num_jogo")),
                "home_team": game.get("mandante", {}).get("nome"),
                "away_team": game.get("visitante", {}).get("nome"),
                "match_date": str(game.get("data", "")).strip(),
                "match_time": str(game.get("hora", "")).strip(),
                "venue": game.get("local"),
            }
            for document in game.get("documentos", []) or []:
                title = document.get("title", "")
                rows.append({
                    **base,
                    "document_title": title,
                    "document_title_normalized": normalize_text(title),
                    "document_url": document.get("url", ""),
                })
    return rows


def _parse_teams_from_slug(slug: str):
    parts = slug.split("-x-", 1)
    if len(parts) == 2:
        home = parts[0].replace("-", " ").title()
        away = parts[1].replace("-", " ").title()
    else:
        home = away = slug.replace("-", " ").title()
    return home, away


def _get_copa_phase_games(session: requests.Session, year: int, phase_id: str):
    url = f"{BASE_URL}/futebol-brasileiro/tabelas/copa-do-brasil/masculino/{year}/{phase_id}"
    try:
        r = session.get(url, timeout=30)
        if r.status_code != 200:
            return []
        pattern = rf'/futebol-brasileiro/jogos/copa-do-brasil/masculino/{year}/([^/"]+)/(\d+)'
        seen = set()
        result = []
        for slug, game_id in re.findall(pattern, r.text):
            if game_id not in seen:
                seen.add(game_id)
                result.append((slug, game_id))
        return result
    except Exception:
        return []


def _get_copa_game_documents(session: requests.Session, year: int, slug: str, game_id: str):
    url = f"{BASE_URL}/futebol-brasileiro/jogos/copa-do-brasil/masculino/{year}/{slug}/{game_id}"
    try:
        r = session.get(url, timeout=30)
        if r.status_code != 200:
            return []
        pairs = re.findall(
            r'\\"url\\":\\"(https://conteudo\.cbf\.com\.br/sumulas/[^\\"]+)\\",\\"title\\":\\"([^\\"]+)\\"',
            r.text,
        )
        return [(title.strip(), doc_url) for doc_url, title in pairs]
    except Exception:
        return []


def collect_documents() -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False
    rows = []
    for competition_slug, comp_info in COMPETITIONS.items():
        comp_type = comp_info.get("type", "league")
        for year in YEARS:
            raw_id = comp_info["ids"].get(year)
            if raw_id is None:
                continue

            if comp_type == "cup":
                phase_ids = raw_id
                print(f"Collecting {competition_slug} {year} ({len(phase_ids)} phases)...")
                for phase_idx, phase_id in enumerate(phase_ids, start=1):
                    games = _get_copa_phase_games(session, year, phase_id)
                    time.sleep(REQUEST_DELAY)
                    for game_num, (slug, game_id) in enumerate(games, start=1):
                        home_team, away_team = _parse_teams_from_slug(slug)
                        docs = _get_copa_game_documents(session, year, slug, game_id)
                        time.sleep(REQUEST_DELAY)
                        for title, doc_url in docs:
                            rows.append({
                                "competition": competition_slug,
                                "competition_name": comp_info["name"],
                                "year": year,
                                "competition_id": phase_id,
                                "round": phase_idx,
                                "game_id": game_id,
                                "game_number": game_num,
                                "home_team": home_team,
                                "away_team": away_team,
                                "match_date": "",
                                "match_time": "",
                                "venue": "",
                                "document_title": title,
                                "document_title_normalized": normalize_text(title),
                                "document_url": doc_url,
                            })
            else:
                max_rounds = comp_info["max_rounds"]
                print(f"Collecting {competition_slug} {year} ({max_rounds} rounds)...")
                for round_number in range(1, max_rounds + 1):
                    url = f"{API_URL}/{raw_id}/rodada/{round_number}/fase"
                    payload = fetch_json(session, url)
                    rows.extend(flatten_documents(
                        payload, year, raw_id, round_number,
                        competition=competition_slug,
                        competition_name=comp_info["name"],
                    ))
                    time.sleep(REQUEST_DELAY)

    data = pd.DataFrame(rows)
    if not data.empty:
        data = data.sort_values(
            ["competition", "year", "round", "game_number", "document_title"]
        ).reset_index(drop=True)
    return data

In [6]:
documents = collect_documents()
documents_index_path = MATCH_REPORTS_PATH / "cbf_documents_2020_2026.csv"
documents.to_csv(documents_index_path, index=False)
documents.head()

,competition,competition_name,year,competition_id,round,game_id,game_number,home_team,away_team,match_date,match_time,venue,document_title,document_title_normalized,document_url
0,copa-do-brasil,Copa do Brasil,2020,1361,1,814689,1,River,Esporte Clube Bahia,,,,Boletim Financeiro,boletim financeiro,https://conteudo.cbf.com.br/sumulas/2020/4241b...
1,copa-do-brasil,Copa do Brasil,2020,1361,1,814689,1,River,Esporte Clube Bahia,,,,Relatório de Jogo,relatorio de jogo,https://conteudo.cbf.com.br/sumulas/2020/4241r...
2,copa-do-brasil,Copa do Brasil,2020,1361,1,814689,1,River,Esporte Clube Bahia,,,,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/4241s...
3,copa-do-brasil,Copa do Brasil,2020,1361,1,814698,2,Frei Paulistano,Remo,,,,Boletim Financeiro,boletim financeiro,https://conteudo.cbf.com.br/sumulas/2020/42410...
4,copa-do-brasil,Copa do Brasil,2020,1361,1,814698,2,Frei Paulistano,Remo,,,,Relatório de Jogo,relatorio de jogo,https://conteudo.cbf.com.br/sumulas/2020/42410...


In [7]:
games = (
    documents.drop_duplicates(subset=["competition", "year", "game_id"])
    .groupby(["competition", "year"])
    .agg(
        games=("game_id", "count"),
        rounds_with_games=("round", "nunique"),
        first_round=("round", "min"),
        last_round=("round", "max"),
    )
    .reset_index()
    .sort_values(["competition", "year"])
)

print(f"{'competition':<20} {'year':>5} {'games':>6} {'rounds':>7} {'first_r':>8} {'last_r':>7}")
print("-" * 58)
for _, r in games.iterrows():
    print(
        f"{r.competition:<20} {r.year:>5} {r.games:>6} "
        f"{r.rounds_with_games:>7} {r.first_round:>8} {r.last_round:>7}"
    )


competition           year  games  rounds  first_r  last_r
----------------------------------------------------------
copa-do-brasil        2020    120       8        1       8
copa-do-brasil        2021    122       7        1       7
copa-do-brasil        2022    122       7        1       7
copa-do-brasil        2023    122       7        1       7
copa-do-brasil        2024    122       7        1       7
copa-do-brasil        2025    122       7        1       7
copa-do-brasil        2026    126       5        1       5
serie-a               2020    380      38        1      38
serie-a               2021    380      38        1      38
serie-a               2022    380      38        1      38
serie-a               2023    380      38        1      38
serie-a               2024    380      38        1      38
serie-a               2025    380      38        1      38
serie-a               2026    157      16        1      16
serie-b               2020    380      38        1      

## **6. Download das Súmulas**

Filtra os documentos com título `"sumula"` e faz o download de cada PDF para `data/raw/match-reports/<competition>/<year>/round-<round>/`. Arquivos já existentes são detectados pelo tamanho do arquivo e ignorados (`status = "exists"`), tornando a célula idempotente.

O índice final `match_reports_index.csv` é a entrada do notebook 1.

In [ ]:
def build_match_report_path(row: pd.Series) -> Path:
    home = slugify(row["home_team"])
    away = slugify(row["away_team"])
    year = int(row["year"])
    round_number = int(row["round"])
    game_number = int(row["game_number"])
    game_id = row["game_id"]
    competition = row.get("competition") or "serie-a"
    filename = f"{year}_r{round_number:02d}_j{game_number:03d}_{home}_x_{away}_{game_id}_sumula.pdf"
    return MATCH_REPORTS_PATH / competition / str(year) / f"round-{round_number:02d}" / filename


def download_pdf(session: requests.Session, url: str, path: Path, max_retries: int = 3) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return "exists"
    temp_path = path.with_suffix(path.suffix + ".tmp")
    last_error = None
    for attempt in range(max_retries):
        try:
            with session.get(url, timeout=60, stream=True) as response:
                response.raise_for_status()
                with temp_path.open("wb") as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            file.write(chunk)
            temp_path.replace(path)
            return "downloaded"
        except requests.RequestException as error:
            last_error = error
            if temp_path.exists():
                temp_path.unlink()
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
    raise RuntimeError(f"Download failed: {url}") from last_error


def collect_match_reports(documents_df: pd.DataFrame) -> pd.DataFrame:
    reports = documents_df[documents_df["document_title_normalized"].eq("sumula")].copy()
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False
    reports["file_path"] = ""
    reports["download_status"] = ""
    for index, row in reports.iterrows():
        file_path = build_match_report_path(row)
        status = download_pdf(session, row["document_url"], file_path)
        reports.loc[index, "file_path"] = str(file_path)
        reports.loc[index, "download_status"] = status
        time.sleep(REQUEST_DELAY)
    return reports.reset_index(drop=True)

In [9]:
match_reports = collect_match_reports(documents)
match_reports_index_path = MATCH_REPORTS_PATH / "match_reports_index.csv"
match_reports.to_csv(match_reports_index_path, index=False)
match_reports["download_status"].value_counts(dropna=False)

download_status
downloaded    5663
Name: count, dtype: int64